## Install colmap-cuda12

In [ ]:
!pip -q install pycolmap-cuda12

## Initialize Paths for sparse/ and images/

In [ ]:
from pathlib import Path
import pycolmap

ROOT = Path("/kaggle/working/project")
# IMAGE_PATH = "/kaggle/input/datasets/jaspreetchhabra/csc-frontface/csc_frontface"
IMAGE_PATH = "/kaggle/input/datasets/jaspreetchhabra/csc-full-view-landscape/images"
SPARSE_PATH = "/kaggle/input/datasets/jaspreetchhabra/csc-fullview-sparse/sparse"
DENSE_PATH = ROOT / "dense"

DENSE_PATH.mkdir(parents=True, exist_ok=True)

## Undistorting the images

In [ ]:
undistort_opts = pycolmap.UndistortCameraOptions()
undistort_opts.max_image_size = 2000

pycolmap.undistort_images(
    output_path=str(DENSE_PATH),
    input_path=str(SPARSE_PATH),
    image_path=str(IMAGE_PATH),
    output_type="COLMAP",
    undistort_options=undistort_opts,
)

## Run Colmap's PatchMatch

In [ ]:
from pathlib import Path
import shutil
import pycolmap

DENSE_PATH = Path("/kaggle/working/project/dense")
FUSED_MODEL_DIR = DENSE_PATH / "fused_model"
FUSED_MODEL_DIR.mkdir(parents=True, exist_ok=True)


pm_opts = pycolmap.PatchMatchOptions()

pm_opts.geom_consistency = False

# Reduce image resolution further
pm_opts.max_image_size = 1200   

# Keep PatchMatch lighter
pm_opts.window_radius = 4
pm_opts.num_samples = 8
pm_opts.num_iterations = 3
pm_opts.cache_size = 4.0

pycolmap.patch_match_stereo(
    workspace_path=str(DENSE_PATH),
    workspace_format="COLMAP",
    options=pm_opts,
)


fusion_opts = pycolmap.StereoFusionOptions()

recon = pycolmap.stereo_fusion(
    output_path=str(FUSED_MODEL_DIR),
    workspace_path=str(DENSE_PATH),
    workspace_format="COLMAP",
    input_type="photometric",   # use this if geom_consistency=False
    options=fusion_opts,
    output_type="bin",
)

ply_path = DENSE_PATH / "fused.ply"
recon.export_PLY(str(ply_path))
print("PLY written to:", ply_path)


stereo_dir = DENSE_PATH / "stereo"
if stereo_dir.exists():
    shutil.rmtree(stereo_dir)
    print("Deleted intermediate stereo directory:", stereo_dir)